# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Build the feature vector
# Define buckets
feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 
    'model_used', 'content_age_days'
]

# Simple imputation for building the vector
df_features = df[feature_cols].copy()
df_features['word_count'] = df_features['word_count'].fillna(df_features['word_count'].median())
df_features['char_count'] = df_features['char_count'].fillna(df_features['char_count'].median())
df_features['search_volume'] = df_features['search_volume'].fillna(0)
df_features['cpc'] = df_features['cpc'].fillna(0)
df_features['model_used'] = df_features['model_used'].fillna('unknown')

# Categorical handling
df_features = pd.get_dummies(df_features, columns=['model_used'], prefix='model')

print("Feature vector built. Shape:", df_features.shape)

Feature vector built. Shape: (30000, 11)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
feature_details = [
    {
        "feature": "search_volume",
        "meaning": "Search demand for the target keyword.",
        "missing": "Filled with 0 (assumes no search demand observed).",
        "type": "Numeric",
        "available": "Before production."
    },
    {
        "feature": "competition",
        "meaning": "Difficulty score to rank for the keyword.",
        "missing": "Assumed 0.",
        "type": "Numeric",
        "available": "Before production."
    },
    {
        "feature": "cpc",
        "meaning": "Cost-per-click bid metric.",
        "missing": "Filled with 0.",
        "type": "Numeric",
        "available": "Before production."
    },
    {
        "feature": "word_count",
        "meaning": "Total word count of the content item.",
        "missing": "Filled with median (approx 2000 words).",
        "type": "Numeric",
        "available": "At production time."
    },
    {
        "feature": "char_count",
        "meaning": "Total character count of the content item.",
        "missing": "Filled with median.",
        "type": "Numeric",
        "available": "At production time."
    },
    {
        "feature": "model_used",
        "meaning": "Generation model identity.",
        "missing": "Filled with 'unknown' (encoded as a category).",
        "type": "Categorical (One-Hot)",
        "available": "At production time."
    },
    {
        "feature": "content_age_days",
        "meaning": "Age of content in days.",
        "missing": "None.",
        "type": "Numeric",
        "available": "At prediction time."
    }
]

for item in feature_details:
    print(f"--- {item['feature']} ---")
    print(f"Meaning: {item['meaning']}")
    print(f"Missingness: {item['missing']}")
    print(f"Type: {item['type']}")
    print(f"Available: {item['available']}\n")

--- search_volume ---
Meaning: Search demand for the target keyword.
Missingness: Filled with 0 (assumes no search demand observed).
Type: Numeric
Available: Before production.

--- competition ---
Meaning: Difficulty score to rank for the keyword.
Missingness: Assumed 0.
Type: Numeric
Available: Before production.

--- cpc ---
Meaning: Cost-per-click bid metric.
Missingness: Filled with 0.
Type: Numeric
Available: Before production.

--- word_count ---
Meaning: Total word count of the content item.
Missingness: Filled with median (approx 2000 words).
Type: Numeric
Available: At production time.

--- char_count ---
Meaning: Total character count of the content item.
Missingness: Filled with median.
Type: Numeric
Available: At production time.

--- model_used ---
Meaning: Generation model identity.
Missingness: Filled with 'unknown' (encoded as a category).
Type: Categorical (One-Hot)
Available: At production time.

--- content_age_days ---
Meaning: Age of content in days.
Missingness: 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
import pandas as pd

# 3. Leakage hunt
# Build the target from the same trend signal used in the data dictionary.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Direct leakage test: the column used to define the label should be a perfect predictor.
down_rule = (df['trend_direction'] == 'down').astype(int)
direct_accuracy = (down_rule == df['is_declining_label']).mean()

# Secondary test: trend_pct is derived from the same recent-vs-prior window.
trend_pct_numeric = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)
trend_pct_corr = trend_pct_numeric.corr(df['is_declining_label'])

print("Label rate (is_declining_label):", round(df['is_declining_label'].mean(), 4))
print("Accuracy of rule 'trend_direction == down':", round(direct_accuracy, 4))
print("Correlation of trend_pct with the label:", round(trend_pct_corr, 4))

print("\nLeakage findings:")
for feature, reason in [
    ("trend_direction", "Directly used to define the label; never a feature."),
    ("trend_pct", "Derived from the same 30-day vs previous 30-day comparison used for the label."),
    ("impressions_last_30d", "Recent-window performance overlaps the target period and is too close to the label.")
]:
    print(f"- {feature}: {reason}")


    

Label rate (is_declining_label): 0.5421
Accuracy of rule 'trend_direction == down': 1.0
Correlation of trend_pct with the label: -0.1313

Leakage findings:
- trend_direction: Directly used to define the label; never a feature.
- trend_pct: Derived from the same 30-day vs previous 30-day comparison used for the label.
- impressions_last_30d: Recent-window performance overlaps the target period and is too close to the label.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [9]:
excluded_features = {
    "trend_direction": "Computed from the same recent-vs-prior window that defines the label.",
    "trend_pct": "Derived from the same trend signal and should not be used as a model input.",
    "avg_position": "A post-hoc performance outcome that is not available before prediction and can leak outcome information.",
    "impressions_last_30d": "Uses the most recent window that overlaps the target period; keep it out of a pre-label feature set."
}

for feature, reason in excluded_features.items():
    print(f"- {feature}: {reason}")

- trend_direction: Computed from the same recent-vs-prior window that defines the label.
- trend_pct: Derived from the same trend signal and should not be used as a model input.
- avg_position: A post-hoc performance outcome that is not available before prediction and can leak outcome information.
- impressions_last_30d: Uses the most recent window that overlaps the target period; keep it out of a pre-label feature set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.